In [25]:
import sys
from pathlib import Path

from truststore import inject_into_ssl

inject_into_ssl()

sys.path.append(str(Path.cwd().parent))

import requests  # noqa: E402
from src.config.settings import IndexerSettings, SnowSettings  # noqa: E402

In [26]:
indexer_settings = IndexerSettings()
snow_settings = SnowSettings()
print(f"indexer_settings: {indexer_settings}")
print(f"snow_settings: {snow_settings}")

# proxies
HTTP_PROXY = indexer_settings.http_proxy
HTTPS_PROXY = indexer_settings.https_proxy
NO_PROXY = indexer_settings.no_proxy

# snow settings
snow_url = snow_settings.servicenow_url
snow_client_id = snow_settings.servicenow_client_id
# secret accessable through snow_settings.servicenow_client_secret

print(snow_settings.token_url)

indexer_settings: http_proxy='http://internet-proxy-client.muenchen.de:80' https_proxy='http://internet-proxy-client.muenchen.de:80' no_proxy='localhost,.svc.cluster.local,.muenchen.de' qdrant_url='https://snow-semantic-squirrel-dev-qdrant.apps.test.capk.muenchen.de/' qdrant_api_key='' qdrant_timeout=100 collection_name='EAKTE_KB_DEV' openai_embedding_model='text-embedding-3-large' openai_api_base='https://ki-proxy-test.muenchen.de' openai_api_key='sk-7PuBA_eFmJcZHETKfB4cQQ' embedding_timeout=10 embedding_max_retries=2 indexing_mode='hybrid' dense_vector_name='dense' sparse_vector_name='sparse' sparse_embedding_model='Qdrant/bm25' sparse_embedding_language='german' fastembed_cache_path='./model_cache' document_chunk_size=1000 document_chunk_overlap=200 indexing_batch_size=20 allow_empty_snapshot=False
snow_settings: servicenow_url='https://lhm.service-now.com/api/sn_km_api/knowledge/articles?kb=7ebcf5acc33a2210d05cf6fe05013198' servicenow_client_id='027496691e91432daa4713bfc99d9006' se

In [27]:
token_url = snow_settings.token_url
session = requests.Session()
data = {
    "grant_type": "client_credentials",
    "client_id": snow_client_id,
    "client_secret": snow_settings.servicenow_client_secret.strip() if snow_settings.servicenow_client_secret else "",
    "scope": snow_settings.servicenow_oauth_scope,
}
headers = {"Content-Type": "application/x-www-form-urlencoded"}
print("Requesting ServiceNow access token")
resp = session.post(token_url, data=data, headers=headers, verify=True, proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY})
print(resp.status_code)

try:
    print(resp.json())
except ValueError:
    print(resp.text)
resp.raise_for_status()
token = resp.json().get("access_token")
if not token:
    raise RuntimeError("ServiceNow OAuth response did not contain an access_token")
print(token)

Requesting ServiceNow access token
200
{'access_token': '6uuWYQOjLDhuDVdiUClDDJjB2swyFpI9JNhioStWnSnFWx8reRUmpp1s3HT13LiRZh0l60LhqUkB5ZtR7PnHIg', 'scope': 'sn_km_api/knowledge.read', 'token_type': 'Bearer', 'expires_in': 1799}
6uuWYQOjLDhuDVdiUClDDJjB2swyFpI9JNhioStWnSnFWx8reRUmpp1s3HT13LiRZh0l60LhqUkB5ZtR7PnHIg


In [28]:
def get_category(article):
    meta_description = article.get("meta_description", "")
    if "eakte-nutzer*innen" in meta_description["value"].lower():
        return "user"
    elif "eakte-fachadministrator*innen" in meta_description["value"].lower():
        return "admin"
    else:
        return "general"

In [29]:
session.headers.update({"Authorization": f"Bearer {token}"})
META_FIELDS = ",".join(
    [
        "kb_category",
        "kb_knowledge_base",
        "author",
        "workflow_state",
        "sys_created_on",
        "sys_updated_on",
        "valid_to",
        "sys_view_count",
        "keywords",
        "meta_description",
    ]
)

params = {
    "limit": snow_settings.servicenow_page_size,
    "fields": META_FIELDS,  # <-- add this (KM API param, not sysparm_fields)
}
data = session.get(snow_url, params=params, verify=True, proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY})
articles = data.json()["result"]["articles"]

for article in articles:
    print(article["number"], article["id"], list(article.keys()))
    # articles are split in generally two scpoes: for users and for admins. this is defined in the meta_description field
    # this is important for the vectordb, because we should be able to distinguish between the two scopes when searching for articles.
    # this enables us to provide more specific search results based on the user's role or needs.
    scope = get_category(article.get("fields", {}))
    print(article.get("fields"))  # <-- this is the check that matters

KB0026284 kb_knowledge:00e32ebbc33acf505c7f449dc00131ef ['link', 'id', 'title', 'snippet', 'score', 'number', 'fields']
{'kb_category': {'display_value': 'eAkte Ablage & Vorgangsbearbeitung', 'name': 'kb_category', 'label': 'Kategorie', 'type': 'reference', 'value': '6261ef0bc33eee1472c9716dc001310e'}, 'kb_knowledge_base': {'display_value': 'eAkte', 'name': 'kb_knowledge_base', 'label': 'Wissensdatenbank', 'type': 'reference', 'value': '7ebcf5acc33a2210d05cf6fe05013198'}, 'author': {'display_value': 'Jennifer Skurka', 'name': 'author', 'label': 'Autor', 'type': 'reference', 'value': '958b6f46db7a29d0266b0dcbd3961994'}, 'workflow_state': {'display_value': 'Veröffentlicht', 'name': 'workflow_state', 'label': 'Workflow-Status', 'type': 'workflow', 'value': 'published'}, 'sys_created_on': {'display_value': '26.08.2026 11:51:03', 'name': 'sys_created_on', 'label': 'Erstellt', 'type': 'glide_date_time', 'value': '2026-08-26 09:51:03'}, 'sys_updated_on': {'display_value': '26.08.2026 11:51:14

In [30]:
fields = ",".join(
    [
        "sys_id",
        "number",
        "short_description",
        "text",
        "kb_knowledge_base",
        "kb_category",
        "topic",
        "category",
        "workflow_state",
        "published",
        "valid_to",
        "sys_created_on",
        "sys_updated_on",
        "author",
        "sys_view_count",
        "meta_description",
        "keywords",
    ]
)

article_url = "https://lhm.service-now.com/api/sn_km_api/knowledge/articles/{}"
full_articles = []

for article in articles:
    r = session.get(
        article_url.format(article["id"].split(":")[1]),
        params={
            "sysparm_fields": fields,
            "sysparm_display_value": "all",  # resolve reference fields
        },
        verify=True,
        proxies={"http": HTTP_PROXY, "https": HTTPS_PROXY},
    )
    r.raise_for_status()
    full_articles.append(r.json()["result"])

print(len(full_articles))

100


In [31]:
print(full_articles[0].keys())

dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'embedded_content'])


In [32]:
from src.loaders.snow_loader import SnowLoader  # noqa: E402

loader = SnowLoader(config=snow_settings)

docs = loader.load_documents()

Skipping ServiceNow article 7ff723b6c31fc350a64d3f3c05013114 because it has no content
Skipping ServiceNow article 8718ebb6c31fc350a64d3f3c05013168 because it has no content
Skipping ServiceNow article 8a473f7ec3d30750a64d3f3c050131a8 because it has no content


In [33]:
from pprint import pprint  # noqa: E402

pprint(docs[0].metadata)
print(len(docs))

{'attachments': [],
 'category': 'eAkte Ablage & Vorgangsbearbeitung',
 'created_at': '2026-08-26T09:51:03',
 'knowledge_base': 'eAkte',
 'language': 'de',
 'number': 'KB0026284',
 'scope': 'user',
 'source': '?id=kb_article_view&sys_kb_id=00e32ebbc33acf505c7f449dc00131ef',
 'source_id': 'snow-kb',
 'sys_id': '00e32ebbc33acf505c7f449dc00131ef',
 'title': '(Produktstandard) eAkte - Schriftstücke erfassen über PDF-Drucker',
 'updated_at': '2026-08-26T09:51:14',
 'valid_to': '2027-08-26'}
106


In [38]:
# Ziel sys_id
target_sys_id = '7ff723b6c31fc350a64d3f3c05013114'

# Filtern der Dokumente anhand des sys_id
filtered_docs = [doc for doc in docs if doc.metadata.get('sys_id') == target_sys_id]

# Ausgabe der gefilterten Dokumente
for doc in filtered_docs:
    print(doc.metadata)
    print(doc.page_content)
    print("doc id:", doc.id)
    print("sys_id:", doc.metadata.get("sys_id"))
    print("content length:", len(doc.page_content))

In [39]:
target_sys_id = "7ff723b6c31fc350a64d3f3c05013114"

detail = loader._article_detail(target_sys_id)

print(detail.keys())
print("content:", type(detail.get("content")), detail.get("content"))
print("text:", type(detail.get("text")), detail.get("text"))
print("short_description:", detail.get("short_description"))

dict_keys(['content', 'template', 'language', 'languages', 'number', 'sys_id', 'short_description', 'display_attachments', 'template_table', 'embedded_content'])
content: <class 'list'> [{'label': 'Antwort', 'content': '<p class="isSelectedEnd">Dieser Artikel richtet sich an eAkte-Nutzer*innen.</p>\r\n<p class="isSelectedEnd">Hier erfahren Sie, wer Ihnen hilft, wenn Sie Probleme bei der Nutzung haben.</p>\r\n<h2>Antwort auf: Wer hilft mir, wenn ich Probleme bei der Nutzung habe?</h2>\r\n<p>Wenn Sie Probleme bei der Nutzung der eAkte haben, wenden Sie sich bitte zunächst an Kolleg*innen mit vertieftem eAkte-Know-How in Ihrem Referat oder Eigenbetrieb, beispielsweise erfahrene eAkte-Nutzer*innen, eAkte-Multiplikator*innen oder -Poweruser*innen.\xa0</p>\r\n<p>Wenn diese Kolleg*innen Ihnen nicht helfen können, melden Sie die <a href="https://it-services.muenchen.de/sp?id&#61;sc_cat_item&amp;sys_id&#61;f2385ce61b76a050e52dfddacd4bcb3e" rel="nofollow">Störung online im IT-Service-Portal</a> 